In [23]:
from pathlib import Path

import pandas as pd
import sklearn
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42

df = pd.read_csv('train.csv')
print(f"Форма датасета: {df.shape[0]} строк × {df.shape[1]} столбцов")
display(df.head())


Форма датасета: 891 строк × 12 столбцов


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


**Анализ**

In [24]:
print("Типы данных:")
print(df.dtypes.to_string())
print("\nПропуски по столбцам:")
print(df.isna().sum().to_string())

print("\nЦелевая переменная Survived:")
counts = df["Survived"].value_counts().sort_index()
display(pd.DataFrame({"Количество": counts, "Доля, %": (counts / len(df) * 100).round(2)}))

assert set(df["Survived"].dropna().unique()) == {0, 1}

Типы данных:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object

Пропуски по столбцам:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2

Целевая переменная Survived:


,Количество,"Доля, %"
Survived,,
0,549,61.62
1,342,38.38


**Разбиение датасета на тренировочную/тестовую выборки**

In [25]:
numeric_features = ["Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Pclass", "Sex", "Embarked"]
features = numeric_features + categorical_features

X = df[features].copy()
y = df["Survived"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"Тренировочная выборка: {len(X_train)}; тестовая: {len(X_test)}")
print(f"Доля выживших на train: {y_train.mean():.3f}; на test: {y_test.mean():.3f}")


def calculate_metrics(y_true, y_pred, proba):
    return {
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, proba),
    }


Тренировочная выборка: 712; тестовая: 179
Доля выживших на train: 0.383; на test: 0.385


**Измерение качества константного предсказания**

In [26]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_proba = baseline.predict_proba(X_test)[:, 1]
baseline_metrics = calculate_metrics(y_test, baseline_pred, baseline_proba)

print("Константное предсказание:", baseline.classes_[baseline.class_prior_.argmax()])
print("Метрики бейзлайна:")
display(pd.DataFrame([baseline_metrics], index=["DummyClassifier"]).round(3))

Константное предсказание: 0
Метрики бейзлайна:


,F1,Accuracy,Precision,Recall,ROC-AUC
DummyClassifier,0.0,0.615,0.0,0.0,0.5


**Предобработка и машинное обучение**

In [27]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        solver="liblinear",
        random_state=RANDOM_STATE,
    )),
])

model.fit(X_train, y_train)
model_pred = model.predict(X_test)
model_proba = model.predict_proba(X_test)[:, 1]
model_metrics = calculate_metrics(y_test, model_pred, model_proba)

print("Метрики логистической регрессии:")
display(pd.DataFrame([model_metrics], index=["LogisticRegression"]).round(3))

Метрики логистической регрессии:


,F1,Accuracy,Precision,Recall,ROC-AUC
LogisticRegression,0.724,0.804,0.793,0.667,0.843


**Измерение качества на отложенной выборке**

In [28]:
comparison = pd.DataFrame(
    [baseline_metrics, model_metrics],
    index=["DummyClassifier", "LogisticRegression"],
)
comparison.index.name = "Модель"
display(comparison.round(3))

print("Матрица ошибок логистической регрессии (строки — факт, столбцы — прогноз):")
display(pd.DataFrame(
    confusion_matrix(y_test, model_pred, labels=[0, 1]),
    index=["Факт: 0", "Факт: 1"],
    columns=["Прогноз: 0", "Прогноз: 1"],
))

print("Подробные метрики по классам:")
print(classification_report(
    y_test, model_pred,
    target_names=["Не выжил (0)", "Выжил (1)"],
    zero_division=0,
))
assert model_metrics["F1"] > baseline_metrics["F1"], "Модель не превзошла бейзлайн по F1"
print("Вывод: LogisticRegression превосходит бейзлайн по основной метрике F1.")

,F1,Accuracy,Precision,Recall,ROC-AUC
Модель,,,,,
DummyClassifier,0.000,0.615,0.000,0.000,0.500
LogisticRegression,0.724,0.804,0.793,0.667,0.843


Матрица ошибок логистической регрессии (строки — факт, столбцы — прогноз):


,Прогноз: 0,Прогноз: 1
Факт: 0,98,12
Факт: 1,23,46


Подробные метрики по классам:
              precision    recall  f1-score   support

Не выжил (0)       0.81      0.89      0.85       110
   Выжил (1)       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179

Вывод: LogisticRegression превосходит бейзлайн по основной метрике F1.


In [29]:
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("random_state:", RANDOM_STATE)

pandas: 2.2.3
scikit-learn: 1.6.1
random_state: 42
